# arcedge — full-SF FTTH design from a declarative config

Runs the complete pipeline end-to-end on San Francisco Overture data:

1. **Model config** (YAML): 1 address point → terminal (≤ 12) → FDH (≤ 512)
   → OLT (≤ 4000), with editable unit costs below.
2. **Compile**: POIs drop onto their nearest street edge (breaking it), the
   facility chain is set up (`arcedge_modelc.py`).
3. **Solve**: one design pass per tier; each tier's opened facilities become
   the next tier's demand-weighted terminals.
4. **Metrics**: per-tier facility counts, cable, cost, timing → a summary
   table + `ftth_sf_metrics.json`.
5. **Output**: per-tier GeoParquet (hubs / cable / drops / assignments) you
   can drop straight into QGIS, plus a quick map preview.

No GPU needed — the design solver is CPU (multi-threaded).
Expect ~10–25 min on a Colab CPU for the full city (54,921 addresses).

In [ ]:
# --- parameters -----------------------------------------------------------
TERMINAL_COST = 500        # per terminal (<= TERMINAL_CAP addresses)
TERMINAL_CAP  = 12
FDH_COST      = 20000      # per fiber distribution hub (<= FDH_CAP addresses)
FDH_CAP       = 512
OLT_COST      = 100000     # per OLT (<= OLT_CAP addresses)
OLT_CAP       = 4000
CABLE_PER_M   = 10         # per meter of edge used, per tier
MAX_ADDRESSES = 0          # 0 = all 54,921; set e.g. 10000 for a quick run
JOINT_ROUNDS  = 2          # joint-feedback rounds (1 = greedy chain)
BRANCH = 'claude/stage-1-implementation-plan-9lv5f3'

In [ ]:
import os
if os.path.exists('/content/arcedge'):
    %cd /content/arcedge
    !git pull
else:
    token = ''  # <-- paste a GitHub token here if the repo is private
    url = f'https://{token}@github.com/fhk/arcedge.git' if token else 'https://github.com/fhk/arcedge.git'
    !git clone --branch {BRANCH} {url} /content/arcedge
    %cd /content/arcedge
!git log --oneline -1
!pip install -q pyyaml pyarrow shapely numpy
!cmake -B build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build -j2 2>&1 | tail -2
!ctest --test-dir build --output-on-failure | tail -3

## Address data

The SF street graph is committed; the Overture *places* GeoParquet is not.
**Upload `places_sf_04_2026.parquet`** (drag it into the Files sidebar or run
this cell), or skip the upload to fall back to the committed 400-address
downtown sample.

In [ ]:
import glob
PLACES = None
cands = glob.glob('/content/*.parquet') + glob.glob('/content/arcedge/*.parquet')
if not cands:
    try:
        from google.colab import files  # noqa
        print('Upload the places parquet now (or press cancel to use the downtown sample):')
        up = files.upload()
        cands = ['/content/' + n for n in up]
    except Exception as e:
        print('upload skipped:', e)
if cands:
    PLACES = cands[0]
    print('using places file:', PLACES)
else:
    print('no parquet -> falling back to the downtown 400-address CSV sample')

In [ ]:
# --- write the model config -------------------------------------------------
import yaml
if PLACES:
    addr_source = {'geoparquet': PLACES}
    if MAX_ADDRESSES:
        addr_source['max_points'] = MAX_ADDRESSES
    streets = 'data/sf_streets.graph'
else:
    addr_source = {'csv': 'examples/sf_dt_pois.csv'}
    streets = 'data/sf_downtown.graph'

model = {
  'model_version': 1,
  'name': 'sf-ftth-colab',
  'layers': {
    'streets':   {'source': {'file': streets}},
    'addresses': {'source': addr_source, 'role': 'terminals'},
  },
  'couplings': [{'name': 'drops', 'from': 'addresses', 'to': 'streets',
                 'method': 'nearest_edge_split', 'snap_m': 0.5}],
  'facilities': {
    'terminal': {'open_cost': TERMINAL_COST, 'capacity': {'hard': TERMINAL_CAP}},
    'fdh':      {'open_cost': FDH_COST,      'capacity': {'hard': FDH_CAP}},
    'olt':      {'open_cost': OLT_COST,      'capacity': {'hard': OLT_CAP}},
  },
  'cable': {'fixed_cost_per_m': CABLE_PER_M},
  'commodities': [{'name': 'service',
                   'assignment': {'from': {'nodes': {'layer': 'addresses'}},
                                  'demand': 1, 'tier': 'terminal'}}],
}
os.makedirs('out', exist_ok=True)
with open('out/sf_ftth_colab.yaml', 'w') as f:
    yaml.safe_dump(model, f, sort_keys=False)
print(open('out/sf_ftth_colab.yaml').read())

## Compile + solve the tier chain

In [ ]:
import time
t0 = time.time()
!python3 scripts/arcedge_modelc.py out/sf_ftth_colab.yaml -o out/sf_ftth --solve --rounds {JOINT_ROUNDS}
WALL_S = time.time() - t0
print(f'\ntotal wall-clock: {WALL_S:.0f} s')

## Metrics

In [ ]:
import json
summary = json.load(open('out/sf_ftth/tiers_summary.json'))
manifest = json.load(open('out/sf_ftth/manifest.json'))
n_addr = manifest['report']['pois']
rows = summary['tiers']
print(f"{'tier':10s} {'facilities':>10} {'cable km':>9} {'cost':>12}")
for t in rows:
    print(f"{t['tier']:10s} {t['hubs']:>10} {t['cable_m']/1000:>9.1f} {t['cost']:>12,.0f}")
total = summary['total_cost']
print('-' * 44)
print(f"{'TOTAL':10s} {'':>10} {sum(t['cable_m'] for t in rows)/1000:>9.1f} {total:>12,.0f}")
print(f'\naddresses served: {n_addr}   cost per address: {total / n_addr:,.0f}')
metrics = dict(addresses=n_addr, tiers=rows, total_cost=total,
               cost_per_address=total / n_addr, wall_s=WALL_S,
               params=dict(terminal=[TERMINAL_COST, TERMINAL_CAP],
                           fdh=[FDH_COST, FDH_CAP], olt=[OLT_COST, OLT_CAP],
                           cable_per_m=CABLE_PER_M))
json.dump(metrics, open('ftth_sf_metrics.json', 'w'), indent=2)
print('\nsaved ftth_sf_metrics.json')

## GeoParquet output (per tier) + map preview

In [ ]:
tier_pois = {1: 'out/sf_ftth/tier0.pois', 2: 'out/sf_ftth/tier1.pois',
             3: 'out/sf_ftth/tier2.pois'}
for i, t in enumerate(summary['tiers'], start=1):
    !python3 scripts/solution_to_geoparquet.py design \
        --graph out/sf_ftth/access.graph --pois {tier_pois[i]} \
        --solution out/sf_ftth/tier{i}.solution \
        --out out/sf_ftth/tier{i}_{t['tier']}.parquet

In [ ]:
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from shapely import from_wkb

fig, ax = plt.subplots(figsize=(11, 11))
styles = {1: ('#bbbbbb', 0.4, 'terminal'), 2: ('#1f77b4', 1.0, 'fdh'),
          3: ('#d62728', 2.0, 'olt')}
for i, t in enumerate(summary['tiers'], start=1):
    tb = pq.read_table(f"out/sf_ftth/tier{i}_{t['tier']}.parquet")
    kinds = tb['kind'].to_pylist()
    geoms = from_wkb(tb['geometry'].to_pylist())
    color, lw, label = styles[i]
    done_line = False
    for k, g in zip(kinds, geoms):
        if k == 'cable':
            xs, ys = g.xy
            ax.plot(xs, ys, color=color, lw=lw,
                    label=(label + ' cable') if not done_line else None)
            done_line = True
    hx = [g.x for k, g in zip(kinds, geoms) if k == 'hub']
    hy = [g.y for k, g in zip(kinds, geoms) if k == 'hub']
    ax.scatter(hx, hy, s=12 * i * i, color=color, zorder=5,
               marker='o^s'[i - 1], label=f"{label} ({len(hx)})")
ax.set_aspect(1.27)  # ~cos(lat) for SF
ax.legend(loc='upper left')
ax.set_title('SF FTTH design: terminal / FDH / OLT tiers')
plt.tight_layout(); plt.show()

In [ ]:
# bundle everything for download
!cp ftth_sf_metrics.json /content/
!cd out/sf_ftth && zip -q /content/ftth_sf_output.zip *.parquet tiers_summary.json
!ls -la /content/ftth_sf_output.zip /content/ftth_sf_metrics.json
print('\nDownload from the Files sidebar: ftth_sf_output.zip + ftth_sf_metrics.json')

## Notes

- Tier placement is greedy per tier (each tier optimizes given the one
  below); the exact joint multi-tier formulation is on the roadmap
  (`docs/model-config.md`).
- Costs/capacities are the parameters cell — rerun from the config cell
  after editing. `MAX_ADDRESSES` subsamples for quick iterations.
- The per-tier parquets carry `kind` (hub / cable / drop / poi), `load`,
  `length_m`, and the serving-hub assignment per address — style `cable`
  by `load` in QGIS to see trunk vs branch structure.